# IDM-VTON Test — Lookzi benchmark pairlar uchun
**Tartib:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
> Alohida Colab session'da oching (CatVTON Colab'dan mustaqil).

In [ ]:
# Cell 1 — GPU tekshirish
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOQ'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Drive mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 3 — IDM-VTON + Lookzi clone
%cd /content
!rm -rf /content/IDM-VTON /content/Lookzi
!git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON
!git clone https://github.com/Mohamed-Kudratov/Lookzi.git /content/Lookzi
%cd /content/IDM-VTON
print("Clone tayyor.")

In [ ]:
# Cell 4 — Dependencies
!pip install -q omegaconf einops transformers==4.46.3 accelerate
!pip install -q diffusers==0.25.0 torchvision opencv-python Pillow huggingface_hub
!pip install -q onnxruntime-gpu  # humanparsing uchun
!pip install -q gradio            # gradio_demo uchun
!pip install -q basicsr facexlib  # realesrgan deps
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.5/index.html

import torch
print(f"torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print("Dependencies tayyor.")

In [ ]:
# Cell 5 — IDM-VTON model yuklab olish va ckpt tuzilishini sozlash
import os, shutil
from huggingface_hub import snapshot_download

DRIVE_IDM    = "/content/drive/MyDrive/Lookzi/hf_models/idm-vton"
IDM_CKP_ROOT = "/content/IDM-VTON/ckpt"

# --- 1. Asosiy IDM-VTON model ---
if not os.path.exists(DRIVE_IDM) or not os.listdir(DRIVE_IDM):
    print("IDM-VTON yuklanmoqda (~8GB, 20-30 daqiqa)...")
    snapshot_download(repo_id="yisol/IDM-VTON", local_dir=DRIVE_IDM)
    print("Yuklandi.")
else:
    print("IDM-VTON Drive'da mavjud.")

def symlink_dir(src, dst):
    if os.path.islink(dst): os.unlink(dst)
    if os.path.exists(dst) and not os.path.islink(dst): shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"  symlink: {dst}")

os.makedirs(IDM_CKP_ROOT, exist_ok=True)
symlink_dir(DRIVE_IDM, f"{IDM_CKP_ROOT}/IDM-VTON")

# --- 2. Humanparsing (ONNX) ---
# IDM-VTON uchun parsing_atr.onnx + parsing_lip.onnx kerak
# Agar yisol/IDM-VTON da bo'lsa:
DRIVE_HP = "/content/drive/MyDrive/Lookzi/hf_models/humanparsing"
os.makedirs(DRIVE_HP, exist_ok=True)

HP_REPO = "levihsu/GarmentDiffusion"  # ONNX parsing modellari shu repoda
hp_files_needed = ["parsing_atr.onnx", "parsing_lip.onnx"]
hp_missing = [f for f in hp_files_needed if not os.path.exists(f"{DRIVE_HP}/{f}")]

if hp_missing:
    print(f"Humanparsing ONNX yuklanmoqda ({HP_REPO})...")
    try:
        snapshot_download(
            repo_id=HP_REPO,
            local_dir=DRIVE_HP,
            allow_patterns=["*.onnx"],
        )
        print("Humanparsing yuklandi.")
    except Exception as e:
        print(f"  XATO ({HP_REPO}): {e}")
        # Zaxira: IDM-VTON repo'da ham bo'lishi mumkin
        for f in hp_files_needed:
            src = f"{DRIVE_IDM}/{f}"
            if os.path.exists(src):
                shutil.copy(src, f"{DRIVE_HP}/{f}")
                print(f"  IDM-VTON repo'dan olindi: {f}")
else:
    print("Humanparsing ONNX Drive'da mavjud.")

# ckpt/humanparsing symlink
hp_local = f"{IDM_CKP_ROOT}/humanparsing"
symlink_dir(DRIVE_HP, hp_local)

# --- 3. OpenPose ---
DRIVE_OP = "/content/drive/MyDrive/Lookzi/hf_models/openpose"
os.makedirs(DRIVE_OP, exist_ok=True)

op_weight = f"{DRIVE_OP}/body_pose_model.pth"
if not os.path.exists(op_weight):
    print("OpenPose yuklanmoqda...")
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(
            repo_id="lllyasviel/ControlNet",
            filename="annotator/ckpts/body_pose_model.pth",
            local_dir=DRIVE_OP,
        )
        import glob
        found = glob.glob(f"{DRIVE_OP}/**/*.pth", recursive=True)
        if found:
            shutil.copy(found[0], op_weight)
        print("OpenPose yuklandi.")
    except Exception as e:
        print(f"  XATO (OpenPose): {e}")
else:
    print("OpenPose Drive'da mavjud.")

op_local = f"{IDM_CKP_ROOT}/openpose/ckpts"
os.makedirs(op_local, exist_ok=True)
op_dst = f"{op_local}/body_pose_model.pth"
if os.path.exists(op_weight) and not os.path.exists(op_dst):
    os.symlink(op_weight, op_dst)
    print(f"  symlink: {op_dst}")

# --- 4. Holat tekshirish ---
print("\n--- Holat ---")
checks = {
    "IDM-VTON UNet":    f"{DRIVE_IDM}/unet",
    "parsing_atr.onnx": f"{DRIVE_HP}/parsing_atr.onnx",
    "parsing_lip.onnx": f"{DRIVE_HP}/parsing_lip.onnx",
    "body_pose_model":  op_weight,
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    print(f"  {'OK' if ok else 'YOQ'}: {name}")
    if not ok: all_ok = False

if all_ok:
    print("\nBarcha model fayllar tayyor. Cell 6 ga o'ting.")
else:
    print("\nBa'zi fayllar topilmadi. Xato xabarini menga yuboring.")

In [ ]:
# Cell 6 — IDM-VTON pipeline yuklash + bitta sinov test
import sys, os, torch, traceback
sys.path.insert(0, '/content/IDM-VTON')

start_tryon = None  # global sifatida belgilaymiz
IDM_READY   = False

print("IDM-VTON gradio_demo yuklanmoqda...")
try:
    from gradio_demo.app import start_tryon
    print("Pipeline yuklandi.")

    # Bitta test
    test_person  = "/content/Lookzi/resource/demo/example/person/men/Yifeng_0.png"
    test_garment = "/content/Lookzi/resource/demo/example/condition/lower/men_pants.png"
    from PIL import Image
    p = Image.open(test_person).convert('RGB')
    g = Image.open(test_garment).convert('RGB')

    print("Bitta sinov render...")
    result, mask = start_tryon(
        {"background": p, "layers": [], "composite": None},
        g, "", True, False, 20, 42
    )
    IDM_READY = True
    print("Sinov muvaffaqiyatli!")

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(12, 5))
    for ax, img, t in zip(axes, [p, g, result], ["Person", "Garment", "IDM-VTON natija"]):
        ax.imshow(img.resize((256, 384))); ax.set_title(t, fontsize=13); ax.axis('off')
    plt.suptitle("Sinov natijasi", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"\nXATO: {e}")
    traceback.print_exc()
    print("\n--- Quyidagi xato xabarini menga yuboring ---")

print(f"\nIDM_READY = {IDM_READY}")

In [ ]:
# Cell 7 — Barcha benchmark pairlar uchun test
# Cell 6 da IDM_READY = True bo'lsa run qiling.

if not IDM_READY or start_tryon is None:
    print("IDM-VTON tayyor emas. Cell 6 xatosini tuzating, keyin qayta run qiling.")
else:
    import json, os, torch
    from PIL import Image
    import matplotlib.pyplot as plt

    LOOKZI_ROOT = "/content/Lookzi"
    OUTPUT_DIR  = "/content/drive/MyDrive/Lookzi/idm_test_results"
    CATVTON_DIR = "/content/drive/MyDrive/Lookzi/eval_logs/outputs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Qaysi pairlar: None = hammasi, yoki masalan ["C04", "C05"]
    TEST_IDS = None

    with open(f"{LOOKZI_ROOT}/benchmark/catalog_pairs.json") as f:
        pairs = json.load(f)["pairs"]
    if TEST_IDS:
        pairs = [p for p in pairs if p["id"] in TEST_IDS]

    print(f"{len(pairs)} ta test.\n")

    for pair in pairs:
        pid          = pair["id"]
        tag          = pair["tag"]
        person_path  = f"{LOOKZI_ROOT}/{pair['person']}"
        garment_path = f"{LOOKZI_ROOT}/{pair['garment']}"
        cloth_type   = pair["cloth_type"]
        save_path    = f"{OUTPUT_DIR}/{pid}_idm.png"

        print(f"{'='*50}")
        print(f"{pid} | {cloth_type} | {tag}")

        idm_result = None
        if os.path.exists(save_path):
            print("  Oldin saqlangan — skip.")
            idm_result = Image.open(save_path)
        else:
            torch.cuda.empty_cache()
            try:
                p_img = Image.open(person_path).convert('RGB')
                g_img = Image.open(garment_path).convert('RGB')
                idm_result, _ = start_tryon(
                    {"background": p_img, "layers": [], "composite": None},
                    g_img, "", True, False, 20, 42
                )
                idm_result.save(save_path)
                print(f"  Saqlandi: {save_path}")
            except Exception as e:
                print(f"  XATO: {e}")

        # Ko'rsatish
        p_img = Image.open(person_path).convert('RGB').resize((256, 384))
        g_img = Image.open(garment_path).convert('RGB').resize((256, 384))
        catvton_path = f"{CATVTON_DIR}/{pid}_result.png"
        has_cat = os.path.exists(catvton_path)

        cols  = [p_img, g_img]
        titles = ["Person", "Garment"]
        if has_cat:
            cols.append(Image.open(catvton_path).convert('RGB').resize((256, 384)))
            titles.append("CatVTON")
        if idm_result:
            cols.append(idm_result.resize((256, 384)))
            titles.append("IDM-VTON")

        fig, axes = plt.subplots(1, len(cols), figsize=(len(cols) * 3.5, 6))
        if len(cols) == 1: axes = [axes]
        fig.suptitle(f"{pid} — {cloth_type} — {tag}", fontsize=11, fontweight='bold')
        for ax, img, t in zip(axes, cols, titles):
            ax.imshow(img); ax.set_title(t); ax.axis('off')
        plt.tight_layout(); plt.show()

    print("\nBarcha testlar tugadi.")
    print(f"Natijalar Drive'da: {OUTPUT_DIR}")